# Rhinoform — certified RB-SR internal held-out final-rerun confirmation

This notebook is the confirmatory follow-up to the post-hoc validation development. It uses the already frozen SHA-ranked 576/70/100 identity split, excludes the previously observed original test identities, retrains matched Ridge/RB-SR and LAMM, and keeps the new 100-identity test locked until all validation-selected model hashes are frozen. RB-SR's hard projection is fixed at attenuation 0.75; it is not retuned here.

The confirmation is internal and retrospective, not external or historically untouched: its identities and some pair outcomes were available during earlier project development. The frozen holdout is nevertheless excluded from every input to this final retraining and cannot be reopened until all final-rerun models are frozen. Report it only as held-out-from-final-retrain evidence.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib, importlib, json, os, shutil, subprocess, sys, time

LIVE_ENV = {**os.environ, 'PYTHONUNBUFFERED': '1'}
def run_live(cmd):
    command = [str(value) for value in cmd]
    print('RUN:', ' '.join(command), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=LIVE_ENV)
    try:
        assert process.stdout is not None
        for line in iter(process.stdout.readline, ''):
            print(line, end='', flush=True)
        return_code = process.wait()
    except KeyboardInterrupt:
        process.terminate(); process.wait(); raise
    finally:
        if process.stdout is not None: process.stdout.close()
    if return_code != 0: raise subprocess.CalledProcessError(return_code, command)
    return subprocess.CompletedProcess(command, return_code)

DRIVE_FYP = Path('/content/drive/MyDrive/FYP final')
REPO = Path('/content/drive/MyDrive/Rhinoform_GitHub_Ready_20260808')
REPO_STR = str(REPO)
if REPO_STR not in sys.path: sys.path.insert(0, REPO_STR)
importlib.invalidate_caches()
_prior_pythonpath = LIVE_ENV.get('PYTHONPATH', '')
LIVE_ENV['PYTHONPATH'] = REPO_STR + (os.pathsep + _prior_pythonpath if _prior_pythonpath else '')
os.environ['PYTHONPATH'] = LIVE_ENV['PYTHONPATH']
RESULTS = DRIVE_FYP / 'results/rbsr_final_rerun_holdout_v1'
PROTOCOL = RESULTS / 'protocol'
POLICY = PROTOCOL / 'FINAL_RERUN_HOLDOUT_POLICY_FREEZE.json'
SPLIT = PROTOCOL / 'final_rerun_holdout_split_manifest.json'
TRAIN_PAIRS = PROTOCOL / 'final_rerun_holdout_train_pairs.json'
LOCAL_FYP = Path('/content/rhinoform_final_rerun_holdout')
LOCAL_DATA = LOCAL_FYP / 'data'
RBSR_OUT = RESULTS / 'rbsr/seed20260609'
LAMM_OUT = RESULTS / 'lamm/seed20260609'
for path in (DRIVE_FYP, REPO, POLICY, SPLIT, TRAIN_PAIRS): assert path.exists(), path
RESULTS.mkdir(parents=True, exist_ok=True)
FINAL_EVIDENCE = RESULTS / 'FINAL_CONFIRMATION_EVIDENCE.json'
TEST_RECEIPTS = [RESULTS/'RBSR_TEST_ACCESS_RECEIPT.json', RESULTS/'LAMM_TEST_ACCESS_RECEIPT.json', RESULTS/'CLASSICAL_TEST_ACCESS_RECEIPT.json']
def bootstrap_valid_sha256_sidecar(path):
    path = Path(path)
    sidecar = path.with_name(path.name + '.sha256.json')
    if not path.is_file() or not sidecar.is_file(): return False
    try:
        record = json.loads(sidecar.read_text(encoding='utf-8'))
        return record.get('sha256') == hashlib.sha256(path.read_bytes()).hexdigest()
    except (OSError, ValueError, TypeError):
        return False
FINAL_RUN_ALREADY_COMPLETE = bootstrap_valid_sha256_sidecar(FINAL_EVIDENCE)
if FINAL_RUN_ALREADY_COMPLETE:
    frozen_result = json.loads(FINAL_EVIDENCE.read_text(encoding='utf-8'))
    assert frozen_result['status'] in {'INTERNAL_FINAL_RERUN_HOLDOUT_PASS', 'INTERNAL_FINAL_RERUN_HOLDOUT_FAILED_PRIMARY_HYPOTHESIS'}
    for receipt_path in TEST_RECEIPTS:
        assert bootstrap_valid_sha256_sidecar(receipt_path), receipt_path
        receipt = json.loads(receipt_path.read_text(encoding='utf-8'))
        assert receipt['status'] == 'COMPLETE' and receipt['test_access_count'] == 1, receipt
print('Persistent outputs:', RESULTS)
print('Completed frozen run available:', FINAL_RUN_ALREADY_COMPLETE)

Mounted at /content/drive
Persistent outputs: /content/drive/MyDrive/FYP final/results/rbsr_final_rerun_holdout_v1
Completed frozen run available: True


## Stage licensed meshes to Colab local storage

Meshes and immutable metadata are copied to `/content` for speed. All checkpoints, chunks, receipts and final evidence remain on Drive.

In [2]:
run_live([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', REPO])
assert (REPO/'rhinoform/repro.py').is_file(), REPO/'rhinoform/repro.py'
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
importlib.invalidate_caches()
local_data_ready = (LOCAL_DATA/'manifest.json').is_file() and (LOCAL_DATA/'meshes').is_dir() and (LOCAL_FYP/'roi').is_dir()
if FINAL_RUN_ALREADY_COMPLETE:
    print('Completed frozen run found; skipping licensed-data rsync.')
elif local_data_ready:
    print('Local licensed data already staged; skipping duplicate rsync:', LOCAL_FYP)
else:
    LOCAL_DATA.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE_FYP / 'data/manifest.json', LOCAL_DATA / 'manifest.json')
    source, destination = DRIVE_FYP / 'data/meshes', LOCAL_DATA / 'meshes'
    destination.mkdir(parents=True, exist_ok=True)
    run_live(['rsync', '-a', '--info=progress2', f'{source}/', f'{destination}/'])
    (LOCAL_FYP / 'roi').mkdir(parents=True, exist_ok=True)
    run_live(['rsync', '-a', f'{DRIVE_FYP / "roi"}/', f'{LOCAL_FYP / "roi"}/'])
    print('Local mesh staging complete:', LOCAL_FYP)

RUN: /usr/bin/python3 -m pip install -q --no-deps -e /content/drive/MyDrive/Rhinoform_GitHub_Ready_20260808
Completed frozen run found; skipping licensed-data rsync.


In [3]:
from pathlib import Path
import importlib
import os
import sys

REPO = Path(
    "/content/drive/MyDrive/Rhinoform_GitHub_Ready_20260808"
)
assert (REPO / "rhinoform/repro.py").is_file()

repo_str = str(REPO)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)

prior_pythonpath = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = (
    repo_str
    + (os.pathsep + prior_pythonpath if prior_pythonpath else "")
)

if "LIVE_ENV" in globals():
    LIVE_ENV["PYTHONPATH"] = os.environ["PYTHONPATH"]

importlib.invalidate_caches()

import rhinoform
from rhinoform.repro import sha256_file

print("PASS: kernel and subprocess both use", REPO)
print("rhinoform loaded from:", rhinoform.__file__)

PASS: kernel and subprocess both use /content/drive/MyDrive/Rhinoform_GitHub_Ready_20260808
rhinoform loaded from: /content/drive/MyDrive/Rhinoform_GitHub_Ready_20260808/rhinoform/__init__.py


In [4]:
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
importlib.invalidate_caches()
from rhinoform.repro import atomic_write_json, sha256_file, sha256_json, valid_sha256_sidecar, validate_torch_artifact, write_sha256_sidecar
for path in (POLICY, SPLIT, TRAIN_PAIRS): assert valid_sha256_sidecar(path), path
policy = json.loads(POLICY.read_text())
split = json.loads(SPLIT.read_text())
assert policy['status'] == 'FROZEN_INTERNAL_FINAL_RERUN_HOLDOUT_POLICY'
assert policy['test_access'] is False and policy['test_access_count'] == 0
assert policy['frozen_rbsr_configuration']['gate_reconstruction_objective'] == 'mean_per_pair_vector_rmse_over_non_landmark_roi_vertices_v1'
assert policy['frozen_rbsr_configuration']['gate_exact_handle_contract'] == 'target_controls_hard_overwrite_for_all_projection_modes_v1'
assert policy['frozen_rbsr_configuration']['zero_residual_ridge_identity_audit_required'] is True
assert sha256_file(SPLIT) == policy['split_manifest_sha256']
assert sha256_file(TRAIN_PAIRS) == policy['train_pair_manifest_sha256']
assert (len(split['train_pool_ids']), len(split['val_ids']), len(split['test_ids'])) == (576, 70, 100)
assert not (set(split['train_pool_ids']) & set(split['val_ids']) or set(split['train_pool_ids']) & set(split['test_ids']) or set(split['val_ids']) & set(split['test_ids']))
run_live([sys.executable, '-m', 'unittest', 'discover', '-s', REPO / 'tests', '-v'])
for path in [REPO/'rhinoform/train.py', REPO/'rhinoform/train_rbsr_gate.py', REPO/'scripts/evaluation/rbsr_gate.py', REPO/'scripts/evaluation/rbsr_ridge_fold_projection.py', REPO/'experiments/lamm/run_lamm_facescape.py']:
    compile(path.read_text(encoding='utf-8'), str(path), 'exec')
print('Final-rerun holdout protocol, hash chain and implementation contracts: PASS')

RUN: /usr/bin/python3 -m unittest discover -s /content/drive/MyDrive/Rhinoform_GitHub_Ready_20260808/tests -v
test_required_classical_baseline_api_is_present (test_baselines_api.ClassicalBaselineApiTests.test_required_classical_baseline_api_is_present) ... ok
test_required_reproduction_helpers_are_present (test_baselines_api.ClassicalBaselineApiTests.test_required_reproduction_helpers_are_present) ... ok
test_all_models_freeze_rejects_changed_evaluation_implementation (test_confirmation_split.ConfirmationSplitTests.test_all_models_freeze_rejects_changed_evaluation_implementation) ... ok
test_all_models_freeze_rejects_missing_or_empty_expected_hash (test_confirmation_split.ConfirmationSplitTests.test_all_models_freeze_rejects_missing_or_empty_expected_hash) ... ok
test_all_models_freeze_requires_exact_declared_hash_chain (test_confirmation_split.ConfirmationSplitTests.test_all_models_freeze_requires_exact_declared_hash_chain) ... ok
test_frozen_classical_configuration_is_complete_and_po

## Train the frozen PCA-64 Ridge/CVAE base

Training and checkpoint selection use only the 576 train and 70 validation identities. The package records the exact split and train-pair manifest hashes. Epoch checkpoints are atomic and resumable.

In [5]:
BASE_DIR = RBSR_OUT / 'base'
BASE_DIR.mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, '-u', '-m', 'rhinoform.train', '--repo', LOCAL_DATA, '--out', BASE_DIR, '--epochs', '180', '--batch-size', '16', '--num-workers', '0', '--hidden', '128', '--latent-dim', '8', '--source-pca-dim', '64', '--delta-pca-dim', '16', '--ridge-lambda', '300', '--lr', '0.001', '--beta', '0.0001', '--dense-weight', '1', '--ctrl-weight', '5', '--edge-weight', '0.1', '--lap-weight', '0', '--strain-weight', '0', '--model-kind', 'cvae', '--alpha-grid', '0,0.1,0.2,0.25,0.3,0.4,0.5,0.6,0.75,1.0', '--kl-warmup', '80', '--eval-every', '10', '--patience', '50', '--seed', '20260609', '--pair-seed', '20260809', '--device', 'cuda', '--use-subunit-features', 'false', '--split-manifest', SPLIT, '--train-pairs-json', TRAIN_PAIRS, '--defer-test-evaluation', '--checkpoint-dir', BASE_DIR/'checkpoints', '--checkpoint-every', '1', '--no-save-dense-predictions']
last = BASE_DIR / 'checkpoints/cvae_last.pt'
if last.is_file() and valid_sha256_sidecar(last): cmd += ['--resume-checkpoint', last]
if FINAL_RUN_ALREADY_COMPLETE:
    print('Completed frozen run found; reusing the frozen base package.')
else:
    run_live(cmd)
BASE = BASE_DIR / 'neural_field_model_package_cvae_ew0p1_lw0.pt'
assert validate_torch_artifact(BASE, required_keys=('args','cvae_state_dict','feature_template_sha256','ridge_cond'), repair_sidecar=False)
import torch
base_package = torch.load(BASE, map_location='cpu', weights_only=False)
assert base_package['split_manifest_sha256'] == sha256_file(SPLIT)
assert base_package['train_pair_manifest_sha256'] == sha256_file(TRAIN_PAIRS)
print('Frozen base:', BASE, sha256_file(BASE))

Completed frozen run found; reusing the frozen base package.
Frozen base: /content/drive/MyDrive/FYP final/results/rbsr_final_rerun_holdout_v1/rbsr/seed20260609/base/neural_field_model_package_cvae_ew0p1_lw0.pt 932db0a879a15ee51d8d6d52fb76d380609ed6ee02f85f8ae78afe9e11e8cfd3


## Fail-fast gate runtime smoke (no holdout access)

Before the 80-epoch run, execute the exact training and raw-evaluation path on two frozen train pairs and two validation pairs for one epoch. Outputs stay in ephemeral `/content`; any import, package-contract, real-mesh, loss, checkpoint or evaluator regression stops here before expensive training.

In [6]:
SMOKE_DIR = Path('/content/rbsr_gate_contract_smoke_v2')
if FINAL_RUN_ALREADY_COMPLETE:
    print('Completed frozen run found; gate smoke was already passed and is not repeated.')
else:
    run_live([sys.executable, '-u', '-m', 'rhinoform.train_rbsr_gate', '--repo', LOCAL_DATA, '--base-model-package', BASE, '--out', SMOKE_DIR/'gate', '--epochs', '1', '--batch-size', '2', '--hidden', '64', '--lr', '0.0005', '--dual-lr', '0.05', '--rho', '10', '--tv-weight', '0.01', '--gate-mean-weight', '0.001', '--orientation-margin', '0', '--strain-threshold', '0.1', '--surrogate-temperature', '0.02', '--surrogate-baseline', 'target', '--orientation-budget-multiplier', '1.0', '--strain-budget-multiplier', '0.8', '--constraint-tolerance', '0.02', '--projection', 'none', '--mode', 'primal_dual', '--orientation-weight', '1', '--strain-weight', '1', '--eval-every', '1', '--seed', '20260609', '--device', 'cuda', '--checkpoint-every', '1', '--max-train-pairs', '2', '--max-val-pairs', '2'])
    SMOKE_GATE = SMOKE_DIR/'gate/rbsr_gate_model.pt'
    assert validate_torch_artifact(SMOKE_GATE, required_keys=('gate_state_dict','base_model_package_sha256','best_validation','deployment_status','reconstruction_objective','exact_handle_contract'), repair_sidecar=False)
    run_live([sys.executable, '-u', REPO/'scripts/evaluation/rbsr_gate.py', '--repo', LOCAL_DATA, '--base-model-package', BASE, '--rbsr-package', SMOKE_GATE, '--split', 'validation', '--out', SMOKE_DIR/'raw_validation', '--batch-size', '2', '--chunk-pairs', '2', '--smoke-max-pairs', '2', '--device', 'cuda'])
    smoke_report = json.loads((SMOKE_DIR/'raw_validation/rbsr_evaluation_validation.json').read_text())
    assert smoke_report['n_pairs'] == 2 and smoke_report['split'] == 'validation'
    print('Fail-fast real-data gate training/evaluation smoke: PASS')

Completed frozen run found; gate smoke was already passed and is not repeated.


## Train the fixed projection-none residual gate

The hyperparameters are copied from the already frozen development configuration. This is one gate training run, not a new grid search. The reconstruction objective is the strict per-pair free-ROI vector RMSE and all nine controls are hard-overwritten under projection-none. The final safety claim comes from the hard fold-set projection, not the soft surrogate feasibility flag.

In [7]:
GATE_DIR = RBSR_OUT / 'gate'
GATE_DIR.mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, '-u', '-m', 'rhinoform.train_rbsr_gate', '--repo', LOCAL_DATA, '--base-model-package', BASE, '--out', GATE_DIR, '--epochs', '80', '--batch-size', '8', '--hidden', '64', '--lr', '0.0005', '--dual-lr', '0.05', '--rho', '10', '--tv-weight', '0.01', '--gate-mean-weight', '0.001', '--orientation-margin', '0', '--strain-threshold', '0.1', '--surrogate-temperature', '0.02', '--surrogate-baseline', 'target', '--orientation-budget-multiplier', '1.0', '--strain-budget-multiplier', '0.8', '--constraint-tolerance', '0.02', '--projection', 'none', '--mode', 'primal_dual', '--orientation-weight', '1', '--strain-weight', '1', '--eval-every', '5', '--seed', '20260609', '--device', 'cuda', '--checkpoint-every', '1']
last = GATE_DIR / 'rbsr_gate_last.pt'
if last.is_file() and valid_sha256_sidecar(last): cmd += ['--resume-checkpoint', last]
if FINAL_RUN_ALREADY_COMPLETE:
    print('Completed frozen run found; reusing the frozen residual gate.')
else:
    run_live(cmd)
GATE = GATE_DIR / 'rbsr_gate_model.pt'
assert validate_torch_artifact(GATE, required_keys=('gate_state_dict','base_model_package_sha256','best_validation','deployment_status','reconstruction_objective','exact_handle_contract'), repair_sidecar=False)
gate_package = torch.load(GATE, map_location='cpu', weights_only=False)
assert gate_package['reconstruction_objective'] == 'mean_per_pair_vector_rmse_over_non_landmark_roi_vertices_v1'
assert gate_package['exact_handle_contract'] == 'target_controls_hard_overwrite_for_all_projection_modes_v1'
print('Frozen residual gate:', GATE, sha256_file(GATE))

Completed frozen run found; reusing the frozen residual gate.
Frozen residual gate: /content/drive/MyDrive/FYP final/results/rbsr_final_rerun_holdout_v1/rbsr/seed20260609/gate/rbsr_gate_model.pt 9185d1baac0934b5fdb437ce451296132eba68a83d8f62d1bafe95a1aaa4f972


## Validation-only hard certificate and freeze

First cache raw validation predictions, then evaluate the exact zero-residual Ridge identity audit plus the single predeclared deployable attenuation 0.75. Zero is an audit/fallback, never a selectable improvement. The selector fails closed unless all 4,830 pairs certify `projected_fold_set ⊆ matched_Ridge_fold_set`, zero reproduces Ridge exactly, and attenuation 0.75 has mean RMSE strictly below Ridge.

In [8]:
RAW_VAL = RBSR_OUT / 'validation_raw'
raw_report_path = RAW_VAL / 'rbsr_evaluation_validation.json'
if not valid_sha256_sidecar(raw_report_path):
    assert not FINAL_RUN_ALREADY_COMPLETE, raw_report_path
    run_live([sys.executable, '-u', REPO/'scripts/evaluation/rbsr_gate.py', '--repo', LOCAL_DATA, '--base-model-package', BASE, '--rbsr-package', GATE, '--split', 'validation', '--out', RAW_VAL, '--batch-size', '16', '--chunk-pairs', '320', '--device', 'cuda', '--save-base-comparators'])
assert valid_sha256_sidecar(raw_report_path), raw_report_path
raw_report = json.loads(raw_report_path.read_text())
source_chunk_root = Path(raw_report['chunking']['chunk_root'])
if not source_chunk_root.is_absolute(): source_chunk_root = raw_report_path.parent / source_chunk_root
PROJECTION_DIR = RBSR_OUT / 'certified_projection_validation'
if not FINAL_RUN_ALREADY_COMPLETE:
    run_live([sys.executable, '-u', REPO/'scripts/evaluation/rbsr_ridge_fold_projection.py', '--repo', LOCAL_DATA, '--base-model-package', BASE, '--rbsr-package', GATE, '--source-validation-report', raw_report_path, '--source-chunk-root', source_chunk_root, '--out', PROJECTION_DIR, '--attenuations', '0,0.75', '--smoothing-steps', '0', '--max-iterations', '64', '--uniform-steps', '1001'])
PROJECTION_FREEZE = PROJECTION_DIR / 'RBSR_CERTIFIED_PROJECTION_FREEZE.json'
if not FINAL_RUN_ALREADY_COMPLETE:
    run_live([sys.executable, '-u', REPO/'scripts/analysis/select_rbsr_certified_projection.py', '--projection-report', PROJECTION_DIR/'rbsr_ridge_fold_projection_validation.json', '--out', PROJECTION_FREEZE])
assert valid_sha256_sidecar(PROJECTION_FREEZE), PROJECTION_FREEZE
projection_freeze = json.loads(PROJECTION_FREEZE.read_text())
assert projection_freeze['status'] == 'FROZEN_CERTIFIED_RIDGE_FOLD_PROJECTION'
assert projection_freeze['selected']['attenuation'] == 0.75
assert projection_freeze['selected']['certificate_rate'] == 1.0
assert projection_freeze['zero_gate_ridge_identity']['passed'] is True
print('Confirmation RB-SR validation freeze:', projection_freeze['selected'])

Confirmation RB-SR validation freeze: {'abs_flip_pct': 0.5556777344581697, 'attenuation': 0.75, 'certificate_rate': 1.0, 'dorsum_rmse': 1.0987452373846536, 'edge_strain_p95': 0.21644458864234653, 'hard_certificate_complete': True, 'label': 'attenuation_0p75', 'landmark_rmse': 0.0, 'max_iterations': 64, 'mean_iterations': 13.379710144927536, 'mean_retention': 0.9917427679998795, 'missed_flip_pct': 1.7257087229807195, 'new_flip_delta_vs_ridge': -0.1027461684785978, 'normal_flip_pct': 0.2513229010517986, 'pair_metrics': 'pair_metrics/pair_metrics_attenuation_0p75_validation.csv', 'pair_metrics_sha256': 'ea9e43012a0a4b11633e9fc658a8da0185914866f044b40a8913640cb7ce24b7', 'projection_diagnostics': 'pair_metrics/projection_diagnostics_attenuation_0p75_validation.csv', 'projection_diagnostics_sha256': '7602961a9b0dca264999a9095114d8d2107940ac8958c5eea215ac6f077f09cb', 'rmse_improvement_vs_ridge': 0.012604328761334216, 'roi_rmse': 1.1123204171016405, 'smoothing_steps': 0, 'status_counts': {'loc

## Retrain LAMM under the identical identity split

LAMM keeps its official architecture and method-native training procedure. The same 576/70/100 identities, ROI, nine controls and strict scorer are used. Test meshes are not loaded during this cell. Atomic epoch checkpoints permit safe overnight resume.

In [9]:
LAMM_COMMIT = '87354c05dec341c6d8dd319665dd52553fb03084'
LAMM_ROOT = Path('/content/LAMM_official')
if FINAL_RUN_ALREADY_COMPLETE:
    assert valid_sha256_sidecar(LAMM_OUT/'ae_best.pt')
    assert valid_sha256_sidecar(LAMM_OUT/'manipulation_best.pt')
    print('Completed frozen run found; reusing both frozen LAMM checkpoints without loading the official training runtime.')
else:
    run_live([sys.executable, '-m', 'pip', 'install', '-q', 'einops==0.6.1', 'timm==0.9.2', 'pyyaml==6.0.2', 'trimesh==3.22.3'])
    if not LAMM_ROOT.exists(): run_live(['git', 'clone', 'https://github.com/michaeltrs/LAMM.git', LAMM_ROOT])
    head = subprocess.check_output(['git','-C',str(LAMM_ROOT),'rev-parse','HEAD'], text=True).strip()
    if head != LAMM_COMMIT:
        run_live(['git','-C',LAMM_ROOT,'fetch','origin',LAMM_COMMIT]); run_live(['git','-C',LAMM_ROOT,'checkout','--detach',LAMM_COMMIT])
    assert subprocess.check_output(['git','-C',str(LAMM_ROOT),'rev-parse','HEAD'], text=True).strip() == LAMM_COMMIT
    run_live([sys.executable, '-u', REPO/'experiments/lamm/run_lamm_facescape.py', '--fyp-root', LOCAL_FYP, '--lamm-root', LAMM_ROOT, '--out', LAMM_OUT, '--split-manifest', SPLIT, '--confirmation-policy', POLICY, '--defer-test-evaluation', '--seed', '20260609', '--ae-epochs', '1500', '--manipulation-epochs', '1500', '--ae-batch-size', '32', '--manipulation-batch-size', '16', '--eval-every', '25', '--checkpoint-every', '1'])
    lamm_training = json.loads((LAMM_OUT/'run_provenance.json').read_text())
    assert lamm_training['status'] == 'TRAINED_VALIDATION_SELECTED_TEST_DEFERRED'
    assert lamm_training['test_access'] is False
    assert lamm_training['training']['validation_objective'] == 'mean_per_pair_vector_rmse_over_non_landmark_roi_vertices_v1'
    assert lamm_training['training']['resume_contract'] == 'data_config_commit_and_implementation_bound_v1'
    assert lamm_training['implementation_sha256'] == sha256_file(REPO/'experiments/lamm/run_lamm_facescape.py')
    print('LAMM frozen without test access:', lamm_training['checkpoints'])

Completed frozen run found; reusing both frozen LAMM checkpoints without loading the official training runtime.


## Freeze all model hashes before test access

This cell records the exact Ridge/CVAE base, gate, hard projection, official LAMM checkpoints, split, policy and implementation hashes. It does not load test meshes.

In [10]:
ALL_MODELS_FREEZE = RESULTS / 'ALL_MODELS_FROZEN_BEFORE_TEST.json'
from rhinoform.confirmation import validate_all_models_freeze, validate_frozen_classical_configuration
receipt_paths = [RESULTS/'RBSR_TEST_ACCESS_RECEIPT.json', RESULTS/'LAMM_TEST_ACCESS_RECEIPT.json', RESULTS/'CLASSICAL_TEST_ACCESS_RECEIPT.json']
existing_receipts = [path for path in receipt_paths if path.exists()]
implementation_paths = [
    REPO/'rhinoform/train.py', REPO/'rhinoform/train_rbsr_gate.py', REPO/'rhinoform/baselines.py',
    REPO/'rhinoform/confirmation.py', REPO/'rhinoform/geometry.py', REPO/'rhinoform/safe_fusion.py',
    REPO/'rhinoform/strict_protocol_patch.py', REPO/'rhinoform/rbsr_calibration.py',
    REPO/'scripts/evaluation/rbsr_gate.py', REPO/'scripts/evaluation/rbsr_ridge_fold_projection.py',
    REPO/'scripts/analysis/select_rbsr_certified_projection.py', REPO/'scripts/evaluation/classical_confirmation.py',
    REPO/'scripts/evaluation/direct_paired_statistics.py', REPO/'experiments/lamm/run_lamm_facescape.py',
]
for path in implementation_paths: assert path.is_file(), path
freeze_expected = {
    'confirmation_policy_sha256': sha256_file(POLICY), 'split_manifest_sha256': sha256_file(SPLIT),
    'rbsr_base_sha256': sha256_file(BASE), 'rbsr_gate_sha256': sha256_file(GATE),
    'rbsr_projection_freeze_sha256': sha256_file(PROJECTION_FREEZE),
    'lamm_ae_sha256': sha256_file(LAMM_OUT/'ae_best.pt'),
    'lamm_manipulation_sha256': sha256_file(LAMM_OUT/'manipulation_best.pt'),
    'lamm_official_commit': LAMM_COMMIT,
}
if existing_receipts:
    assert valid_sha256_sidecar(ALL_MODELS_FREEZE), ALL_MODELS_FREEZE
    payload = json.loads(ALL_MODELS_FREEZE.read_text(encoding='utf-8'))
    validate_all_models_freeze(payload, expected=freeze_expected, expected_implementations=payload['implementation_hashes'])
    if not FINAL_RUN_ALREADY_COMPLETE:
        current_implementations = {path.relative_to(REPO).as_posix(): sha256_file(path) for path in implementation_paths}
        assert current_implementations == payload['implementation_hashes'], 'Cannot resume test access with changed implementations'
    print('PASS: existing pre-test freeze reused; it was not rebuilt after test access.')
else:
    policy = json.loads(POLICY.read_text(encoding='utf-8'))
    classical_configuration = validate_frozen_classical_configuration(policy['frozen_classical_configuration'])
    lamm_training_provenance_path = LAMM_OUT / 'run_provenance.json'
    lamm_training_provenance = json.loads(lamm_training_provenance_path.read_text(encoding='utf-8'))
    assert lamm_training_provenance['status'] == 'TRAINED_VALIDATION_SELECTED_TEST_DEFERRED'
    for path in [POLICY, SPLIT, BASE, GATE, PROJECTION_FREEZE, LAMM_OUT/'ae_best.pt', LAMM_OUT/'manipulation_best.pt', lamm_training_provenance_path]:
        assert valid_sha256_sidecar(path), path
    implementation_hashes = {path.relative_to(REPO).as_posix(): sha256_file(path) for path in implementation_paths}
    payload = {
        'status': 'ALL_MATCHED_MODELS_FROZEN_BEFORE_TEST', 'test_access': False,
        **freeze_expected,
        'lamm_deferred_training_provenance_sha256': sha256_file(lamm_training_provenance_path),
        'lamm_training_implementation_sha256': lamm_training_provenance['implementation_sha256'],
        'classical_configuration_sha256': sha256_json(classical_configuration),
        'implementation_hashes': implementation_hashes,
    }
    atomic_write_json(ALL_MODELS_FREEZE, payload); write_sha256_sidecar(ALL_MODELS_FREEZE)
    assert valid_sha256_sidecar(ALL_MODELS_FREEZE)
    validate_all_models_freeze(payload, expected={
        **freeze_expected,
        'lamm_deferred_training_provenance_sha256': sha256_file(lamm_training_provenance_path),
        'lamm_training_implementation_sha256': lamm_training_provenance['implementation_sha256'],
    }, expected_implementations=implementation_hashes)
    print('PASS: pre-test freeze rebuilt with', len(implementation_hashes), 'implementation hashes')
    print('No test receipt exists; one-shot holdout remains unconsumed.')

PASS: existing pre-test freeze reused; it was not rebuilt after test access.


## One-shot final-rerun holdout

Run this only after the previous cell passes. Each method writes a receipt before loading a test mesh; interruption is resumable only for the identical hash chain. A completed receipt prevents a changed model from masquerading as the same one-shot run.

In [11]:
RUN_FINAL_HOLDOUT = True
RBSR_TEST = RBSR_OUT / 'one_shot_test'
CLASSICAL_OUT = RESULTS / 'classical'
if RUN_FINAL_HOLDOUT and FINAL_RUN_ALREADY_COMPLETE:
    print('Final-rerun holdout already COMPLETE; verified receipts and reusing immutable test outputs.')
elif RUN_FINAL_HOLDOUT:
    assert valid_sha256_sidecar(ALL_MODELS_FREEZE)
    run_live([sys.executable, '-u', REPO/'scripts/evaluation/rbsr_gate.py', '--repo', LOCAL_DATA, '--base-model-package', BASE, '--rbsr-package', GATE, '--projection-freeze', PROJECTION_FREEZE, '--confirmation-policy', POLICY, '--test-access-receipt', RESULTS/'RBSR_TEST_ACCESS_RECEIPT.json', '--all-models-freeze', ALL_MODELS_FREEZE, '--split', 'test', '--out', RBSR_TEST, '--batch-size', '16', '--chunk-pairs', '320', '--device', 'cuda', '--save-base-comparators'])
    run_live([sys.executable, '-u', REPO/'experiments/lamm/run_lamm_facescape.py', '--fyp-root', LOCAL_FYP, '--lamm-root', LAMM_ROOT, '--out', LAMM_OUT, '--split-manifest', SPLIT, '--confirmation-policy', POLICY, '--test-access-receipt', RESULTS/'LAMM_TEST_ACCESS_RECEIPT.json', '--all-models-freeze', ALL_MODELS_FREEZE, '--seed', '20260609', '--ae-epochs', '1500', '--manipulation-epochs', '1500', '--ae-batch-size', '32', '--manipulation-batch-size', '16', '--eval-every', '25', '--checkpoint-every', '1'])
    run_live([sys.executable, '-u', REPO/'scripts/evaluation/classical_confirmation.py', '--repo', LOCAL_DATA, '--base-model-package', BASE, '--split-manifest', SPLIT, '--confirmation-policy', POLICY, '--all-models-freeze', ALL_MODELS_FREEZE, '--test-access-receipt', RESULTS/'CLASSICAL_TEST_ACCESS_RECEIPT.json', '--out', CLASSICAL_OUT, '--chunk-pairs', '160'])
    print('One-shot matched tests complete for RB-SR, Ridge/CVAE/Hybrid, LAMM and the three classical baselines. Do not tune or replace the frozen operating point.')
else:
    print('Final-rerun holdout remains locked: RUN_FINAL_HOLDOUT=False')

Final-rerun holdout already COMPLETE; verified receipts and reusing immutable test outputs.


## Direct paired inference and immutable final evidence

After the one-shot test, this cell performs direct identity-clustered paired inference for certified RB-SR versus matched Ridge, LAMM, CVAE, Hybrid, Laplacian, bi-Laplacian and ARAP. It records the held-out outcome whether the primary hypothesis passes or fails; it never changes the frozen model or operating point.

In [12]:
if RUN_FINAL_HOLDOUT and FINAL_RUN_ALREADY_COMPLETE:
    assert valid_sha256_sidecar(FINAL_EVIDENCE), FINAL_EVIDENCE
    final_evidence = json.loads(FINAL_EVIDENCE.read_text(encoding='utf-8'))
    print('PASS: immutable final evidence already exists; no model, holdout, or bootstrap computation was repeated.')
    print(json.dumps(final_evidence, indent=2))
elif RUN_FINAL_HOLDOUT:
    from rhinoform.repro import atomic_copy_file
    PAIR_DIR = RESULTS / 'direct_paired_statistics/pair_metrics'
    STATS_DIR = RESULTS / 'direct_paired_statistics'
    PAIR_DIR.mkdir(parents=True, exist_ok=True)
    frozen_models = json.loads(ALL_MODELS_FREEZE.read_text())
    assert frozen_models['implementation_hashes']['scripts/evaluation/direct_paired_statistics.py'] == sha256_file(REPO/'scripts/evaluation/direct_paired_statistics.py')
    atomic_copy_file(RBSR_TEST/'pair_metrics_ridge_sourcepca_clean_test.csv', PAIR_DIR/'identity_bootstrap_pair_metrics_ridge.csv')
    atomic_copy_file(RBSR_TEST/'pair_metrics_rbsr_test.csv', PAIR_DIR/'identity_bootstrap_pair_metrics_certified_rbsr.csv')
    atomic_copy_file(LAMM_OUT/'identity_bootstrap_pair_metrics_lamm.csv', PAIR_DIR/'identity_bootstrap_pair_metrics_lamm.csv')
    atomic_copy_file(RBSR_TEST/'pair_metrics_cvae_clean_test.csv', PAIR_DIR/'identity_bootstrap_pair_metrics_cvae.csv')
    atomic_copy_file(RBSR_TEST/'pair_metrics_hybrid_validation_selected_clean_test.csv', PAIR_DIR/'identity_bootstrap_pair_metrics_hybrid.csv')
    for method in ('laplacian', 'bilaplacian', 'arap'):
        atomic_copy_file(CLASSICAL_OUT/f'identity_bootstrap_pair_metrics_{method}.csv', PAIR_DIR/f'identity_bootstrap_pair_metrics_{method}.csv')
    run_live([sys.executable, '-u', REPO/'scripts/evaluation/direct_paired_statistics.py', '--pair-dir', PAIR_DIR, '--baseline', 'ridge', '--methods', 'certified_rbsr', '--out', STATS_DIR/'paired_primary_rbsr_vs_ridge.csv', '--seed', '20260609', '--n-boot', '10000'])
    run_live([sys.executable, '-u', REPO/'scripts/evaluation/direct_paired_statistics.py', '--pair-dir', PAIR_DIR, '--baseline', 'ridge', '--methods', 'certified_rbsr,lamm,cvae,hybrid,laplacian,bilaplacian,arap', '--out', STATS_DIR/'paired_all_vs_ridge.csv', '--seed', '20260609', '--n-boot', '10000'])
    run_live([sys.executable, '-u', REPO/'scripts/evaluation/direct_paired_statistics.py', '--pair-dir', PAIR_DIR, '--baseline', 'lamm', '--methods', 'certified_rbsr', '--out', STATS_DIR/'paired_rbsr_vs_lamm.csv', '--seed', '20260609', '--n-boot', '10000'])
    expected_stats_json_rows = {'paired_primary_rbsr_vs_ridge': 6, 'paired_all_vs_ridge': 42, 'paired_rbsr_vs_lamm': 6}
    for stem, expected_rows in expected_stats_json_rows.items():
        csv_path, json_path = STATS_DIR/f'{stem}.csv', STATS_DIR/f'{stem}.json'
        assert valid_sha256_sidecar(csv_path), csv_path
        stats_payload = json.loads(json_path.read_text(encoding='utf-8'))
        assert isinstance(stats_payload.get('rows'), list) and len(stats_payload['rows']) == expected_rows, json_path
        if not valid_sha256_sidecar(json_path): write_sha256_sidecar(json_path)
        assert valid_sha256_sidecar(json_path), json_path
        print('PASS: paired-statistics JSON sidecar present:', json_path.name, 'rows=', expected_rows)
    rbsr_report_path = RBSR_TEST/'rbsr_evaluation_test.json'
    lamm_report_path = LAMM_OUT/'lamm_test_summary.json'
    classical_report_path = CLASSICAL_OUT/'classical_confirmation_test_summary.json'
    rbsr_report = json.loads(rbsr_report_path.read_text())
    ridge_summary = rbsr_report['clean_base_comparators']['ridge_sourcepca_clean']['summary']
    rbsr_summary = rbsr_report['summary']
    certificate = rbsr_report['projection_certificate']
    headline_pass = bool(rbsr_summary['roi_rmse'] < ridge_summary['roi_rmse'] and rbsr_summary['normal_flip_pct'] <= ridge_summary['normal_flip_pct'] and certificate['certificate_rate'] == 1.0)
    evidence_inputs = [ALL_MODELS_FREEZE, RESULTS/'RBSR_TEST_ACCESS_RECEIPT.json', RESULTS/'LAMM_TEST_ACCESS_RECEIPT.json', RESULTS/'CLASSICAL_TEST_ACCESS_RECEIPT.json', rbsr_report_path, lamm_report_path, classical_report_path, STATS_DIR/'paired_primary_rbsr_vs_ridge.csv', STATS_DIR/'paired_primary_rbsr_vs_ridge.json', STATS_DIR/'paired_all_vs_ridge.csv', STATS_DIR/'paired_all_vs_ridge.json', STATS_DIR/'paired_rbsr_vs_lamm.csv', STATS_DIR/'paired_rbsr_vs_lamm.json']
    for path in evidence_inputs: assert valid_sha256_sidecar(path), path
    for receipt_path in (RESULTS/'RBSR_TEST_ACCESS_RECEIPT.json', RESULTS/'LAMM_TEST_ACCESS_RECEIPT.json', RESULTS/'CLASSICAL_TEST_ACCESS_RECEIPT.json'):
        receipt = json.loads(receipt_path.read_text()); assert receipt['status'] == 'COMPLETE' and receipt['test_access_count'] == 1, receipt
    final_evidence = {
        'status': 'INTERNAL_FINAL_RERUN_HOLDOUT_PASS' if headline_pass else 'INTERNAL_FINAL_RERUN_HOLDOUT_FAILED_PRIMARY_HYPOTHESIS',
        'claim_boundary': 'internal identity-disjoint held-out-from-final-retrain evidence; not historically untouched and not external',
        'predeclared_primary_hypothesis': 'certified RB-SR test ROI RMSE < matched Ridge and new flip <= matched Ridge with certificate rate 1.0',
        'headline_pass': headline_pass, 'test_access_count': 1, 'post_test_tuning_permitted': False,
        'rbsr_summary': rbsr_summary, 'matched_ridge_summary': ridge_summary,
        'lamm_summary': json.loads(lamm_report_path.read_text())['means'],
        'classical_summaries': {method: record['summary'] for method, record in json.loads(classical_report_path.read_text())['methods'].items()},
        'projection_certificate': certificate,
        'artifact_sha256': {str(path.relative_to(RESULTS)): sha256_file(path) for path in evidence_inputs},
    }
    atomic_write_json(FINAL_EVIDENCE, final_evidence); write_sha256_sidecar(FINAL_EVIDENCE)
    print(json.dumps(final_evidence, indent=2))
else:
    print('Final paired statistics remain locked until the one-shot final-rerun holdout is explicitly enabled.')

PASS: immutable final evidence already exists; no model, holdout, or bootstrap computation was repeated.
{
  "artifact_sha256": {
    "ALL_MODELS_FROZEN_BEFORE_TEST.json": "5787b767ee5138a0dfefe644afc266ff47acb0427b8ec00b792a4ab8355a429b",
    "CLASSICAL_TEST_ACCESS_RECEIPT.json": "cd5614de67d3995c8e9a4ba967003c0f484bd2debe1c6a27c0397622b844ae2f",
    "LAMM_TEST_ACCESS_RECEIPT.json": "da160f8c05e44b8d092d127b606f0056405e3c0ea2c75c82d64e0e34f83cf9ef",
    "RBSR_TEST_ACCESS_RECEIPT.json": "bb7b8bde80f5ded28b268d611f08df324e8db39585cd704e74b04340ed9b8a2f",
    "classical/classical_confirmation_test_summary.json": "c8ab7b07f7dffea1328621fd18cca38ac1273c98251b9b74a65eb4aca116bd6b",
    "direct_paired_statistics/paired_all_vs_ridge.csv": "a7652334442c28060482a72c7b95aa2279278aa6814008969c4f4a7bbdb1712e",
    "direct_paired_statistics/paired_all_vs_ridge.json": "82ce2b17539c05b34d83d4b2aadec7712b52e0afdd4791dd52e3638f111bebd5",
    "direct_paired_statistics/paired_primary_rbsr_vs_ridge.csv": 